# Understand Metrics Pipeline
Step-by-step notebook for running SciTools Understand metrics on SWE-Bench Pro patches.

## Step 1 — Understand `und` Base Setup
Configure paths, set up the environment variables required by `und`, and define the command runner.

In [44]:
TARGET_INSTANCE = ""   # pin to one instance ID for testing; "" = all
TARGET_REPO     = "" 

In [32]:
import os
import subprocess
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────
SCI_BIN = "/home/AV00500/HULLM/scitools/bin/linux64"   # SciTools bin directory
UND_EXE = os.path.join(SCI_BIN, "und")                 # und executable

UDB_DIR = Path("understand_dbs")                        # where .und databases are stored
UDB_DIR.mkdir(parents=True, exist_ok=True)

assert Path(UND_EXE).exists(), f"und not found at: {UND_EXE}"
print(f"[OK] und found: {UND_EXE}")


def get_und_env() -> dict:
    """Return os.environ with LD_LIBRARY_PATH pointing at SciTools libs."""
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = SCI_BIN + ":" + env.get("LD_LIBRARY_PATH", "")
    env["PERL5LIB"] = ""
    env["PERLLIB"]  = ""
    return env


def run_und(args: list, timeout: int = 300) -> subprocess.CompletedProcess:
    """
    Run an und sub-command and print the step.
    args: list of arguments AFTER the und executable,
          e.g. ["create", "-languages", "Python", str(udb_path)]
    """
    cmd   = [UND_EXE] + [str(a) for a in args]
    label = args[0] if args else "und"
    print(f"  [und {label}] {' '.join(str(a) for a in args[1:])[:120]}")

    result = subprocess.run(
        cmd, env=get_und_env(),
        text=True, capture_output=True, timeout=timeout,
    )
    if result.returncode == 0:
        print(f"  → OK")
    else:
        print(f"  → FAILED (rc={result.returncode})")
        if result.stderr:
            print(f"  stderr: {result.stderr.strip()[:300]}")
    return result


# Smoke-test
run_und(["version"])

[OK] und found: /home/AV00500/HULLM/scitools/bin/linux64/und
  [und version] 
  → OK


CompletedProcess(args=['/home/AV00500/HULLM/scitools/bin/linux64/und', 'version'], returncode=0, stdout='(Build 1241)\n', stderr='')

## Step 2 — Load Dataset
Load the SWE-Bench Pro parquet and collect valid instance folders from the eval directory.

In [ ]:
import re
import pandas as pd

# ── Config ─────────────────────────────────────────────────────────────────
MODEL    = "claude-45sonnet-10132025"
EVAL_DIR = Path("/home/AV00500/Issam/SWE-Bench_Pro/swebench_pro_pipeline") / "data" / MODEL / "eval"


REPO_MAP = {
    "ansible__ansible":             "ansible",
    "qutebrowser__qutebrowser":     "qutebrowser",
    "internetarchive__openlibrary": "openlibrary",
}
REPO_LANG = {
    "ansible__ansible":             "Python",
    "qutebrowser__qutebrowser":     "Python",
    "internetarchive__openlibrary": "Python",
}
LANG_EXT = {
    "Python":     {".py"},
    "TypeScript": {".ts", ".tsx", ".js", ".jsx"},
    "Go":         {".go"},
}

# ── Load SWE-Bench Pro parquet ─────────────────────────────────────────────
print("Step 2 — Loading dataset...")
df_dataset = pd.read_parquet(
    "hf://datasets/ScaleAI/SWE-bench_Pro/data/test-00000-of-00001.parquet"
)
commit_map = dict(zip(df_dataset["instance_id"], df_dataset["base_commit"]))
patch_map  = dict(zip(df_dataset["instance_id"], df_dataset["patch"]))
print(f"  → {len(commit_map)} instances loaded from dataset")

# ── Collect instance folders ───────────────────────────────────────────────
assert EVAL_DIR.is_dir(), f"Eval dir not found: {EVAL_DIR}"
instances = [p for p in EVAL_DIR.iterdir() if p.is_dir()]
if TARGET_INSTANCE:
    instances = [p for p in instances if p.name == TARGET_INSTANCE]
print(f"  → {len(instances)} instance folders found in eval dir")

# ── Build flat list of valid instances ────────────────────────────────────
_INST_RE = re.compile(r"^instance_(.+)-[0-9a-f]{40}-v", re.I)

valid_instances = []
for inst_dir in instances:
    m = _INST_RE.match(inst_dir.name)
    if not m:
        continue
    repo_id = m.group(1)
    if repo_id not in REPO_MAP:
        continue
    if TARGET_REPO and repo_id != TARGET_REPO:
        continue
    iid         = inst_dir.name
    base_commit = commit_map.get(iid)
    if not base_commit:
        continue
    valid_instances.append({
        "inst_dir":        inst_dir,
        "iid":             iid,
        "repo_id":         repo_id,
        "repo_name":       REPO_MAP[repo_id],
        "lang":            REPO_LANG[repo_id],
        "base_commit":     base_commit,
        "gold_patch_text": patch_map.get(iid, ""),
        "llm_patch_path":  inst_dir / "_patch.diff",
    })

print(f"  → {len(valid_instances)} valid instances after filtering")
for inst in valid_instances[:5]:
    print(f"  {inst['iid']}  |  repo={inst['repo_name']}  |  commit={inst['base_commit'][:10]}...")

Step 2 — Loading dataset...
  → 731 instances loaded from dataset
  → 1 instance folders found in eval dir
  → 1 valid instances after filtering
  instance_qutebrowser__qutebrowser-a25e8a09873838ca9efefd36ea8a45170bbeb95c-vc2f56a753b62a190ddb23cd330c257b9cf560d12  |  repo=qutebrowser  |  commit=83bef2ad4b...


## Step 3 — Create Base UDB and Copy to Gold / LLM

The base UDB is built once at the base commit with a full analysis.
It is then **copied** to two independent databases:
- `repo_gold.und` — used for the gold patch
- `repo_llm.und`  — used for the LLM patch

This eliminates the need to revert patches and re-analyze after each one.

In [36]:
import shutil

REPO_ROOT = Path("/home/AV00500/Issam/SWE-Bench_Pro/repos")
REPO_ROOT.mkdir(parents=True, exist_ok=True)

OUT_DIR = Path(f"understand_results/{MODEL}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

UDB_SETTINGS = {
    "-MetricShowDeclaredInFile":                "On",
    "-MetricFileNameDisplayMode":               "RelativePath",
    "-MetricShowFunctionParameterTypes":        "On",
    "-MetricAddUniqueNameColumn":               "On",
    "-MetricCyclomatic":                        "All",
    "-MetricShowEssential":                     "On",
    "-MetricShowPaths":                         "On",
    "-MetricShowMaximumNesting":                "On",
    "-MetricShowCouplingAndCohesionMetrics":    "On",
    "-MetricShowInheritanceMetrics":            "On",
    "-MetricShowMethodAndVariableCountMetrics": "On",
    "-MetricShowLineCountMetrics":              "On",
    "-MetricShowStatementCountMetrics":         "On",
    "-MetricShowAggregatedFunctionMetrics":     "On",
    "-MetricShowAggregatedClassMetrics":        "On",
    "-MetricShowAggregatedFileMetrics":         "On",
}


def clone_repo_if_needed(repo_id, repo_name):
    """Clone the GitHub repository if not already present locally."""
    repo_path = REPO_ROOT / repo_name
    if repo_path.is_dir():
        print(f"  [clone] already exists: {repo_path}")
        return repo_path
    owner, name = repo_id.split("__")
    url = f"https://github.com/{owner}/{name}.git"
    print(f"  [clone] cloning {url}")
    r = subprocess.run(["git", "clone", url, str(repo_path)],
                       text=True, capture_output=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")
    print(f"  [clone] OK")
    return repo_path


def checkout_commit(repo_path, commit):
    """Hard-reset the repo to a specific base commit."""
    print(f"  [git] checkout {commit[:10]}...")
    for cmd in (
        ["git", "reset", "--hard"],
        ["git", "clean", "-fd"],
        ["git", "checkout", commit],
    ):
        r = subprocess.run(cmd, cwd=repo_path, text=True, capture_output=True)
        if r.returncode != 0:
            raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    print(f"  [git] OK")


def create_base_udb(udb_path, repo_path, lang):
    """
    Create a fresh .und database, add the full repo, and run a complete
    analysis. This is the baseline UDB built once per instance.
    """
    print(f"  [und] creating base UDB: {udb_path.name}  (lang={lang})")
    shutil.rmtree(str(udb_path), ignore_errors=True)
    udb_path.with_suffix(".csv").unlink(missing_ok=True)

    run_und(["create", "-languages", lang, str(udb_path)])

    settings_args = ["settings"]
    for k, v in UDB_SETTINGS.items():
        settings_args += [k, v]
    settings_args.append(str(udb_path))
    run_und(settings_args)

    run_und(["add", str(repo_path), str(udb_path)])
    print(f"  [analyze] full analysis (-all)...")
    run_und(["analyze", "-all", str(udb_path)], timeout=600)
    print(f"  [OK] base UDB ready: {udb_path}")


def copy_udb(src_udb, dst_udb):
    """
    Copy a .und database directory to a new path.
    Understand databases are directories, so we use shutil.copytree.
    Any existing destination is removed first.
    """
    shutil.rmtree(str(dst_udb), ignore_errors=True)
    dst_udb.with_suffix(".csv").unlink(missing_ok=True)
    shutil.copytree(str(src_udb), str(dst_udb))
    print(f"  [copy_udb] {src_udb.name} → {dst_udb.name}")


def init_patch_udbs(repo_name, repo_path, lang):
    """
    Build the base UDB once, then copy it to two independent databases:
      - <repo_name>_gold.und  for the gold patch
      - <repo_name>_llm.und   for the LLM patch

    Returns (udb_gold, udb_llm) paths.
    """
    base_udb = UDB_DIR / f"{repo_name}_base.und"
    gold_udb = UDB_DIR / f"{repo_name}_gold.und"
    llm_udb  = UDB_DIR / f"{repo_name}_llm.und"

    print(f"\nStep 3 — Building base UDB for: {repo_name}")
    create_base_udb(base_udb, repo_path, lang)

    print(f"\nStep 3 — Copying base UDB to gold and llm copies")
    copy_udb(base_udb, gold_udb)
    copy_udb(base_udb, llm_udb)

    return gold_udb, llm_udb


# ── Smoke-test on first instance ──────────────────────────────────────────
inst      = valid_instances[0]
repo_path = clone_repo_if_needed(inst["repo_id"], inst["repo_name"])
checkout_commit(repo_path, inst["base_commit"])
gold_udb, llm_udb = init_patch_udbs(inst["repo_name"], repo_path, inst["lang"])

  [clone] cloning https://github.com/qutebrowser/qutebrowser.git
  [clone] OK
  [git] checkout 83bef2ad4b...
  [git] OK

Step 3 — Building base UDB for: qutebrowser
  [und] creating base UDB: qutebrowser_base.und  (lang=Python)
  [und create] -languages Python understand_dbs/qutebrowser_base.und
  → OK
  [und settings] -MetricShowDeclaredInFile On -MetricFileNameDisplayMode RelativePath -MetricShowFunctionParameterTypes On -MetricAddUniq
  → OK
  [und add] /home/AV00500/Issam/SWE-Bench_Pro/repos/qutebrowser understand_dbs/qutebrowser_base.und
  → OK
  [analyze] full analysis (-all)...
  [und analyze] -all understand_dbs/qutebrowser_base.und
  → OK
  [OK] base UDB ready: understand_dbs/qutebrowser_base.und

Step 3 — Copying base UDB to gold and llm copies
  [copy_udb] qutebrowser_base.und → qutebrowser_gold.und
  [copy_udb] qutebrowser_base.und → qutebrowser_llm.und


## Step 4 — Detect Changed Files from a Patch
Parse the patch to get the list of source files it touches, filtered by language extension.

In [37]:
import tempfile

def get_changed_files(repo_path, patch_text, exts):
    """
    Return the list of source files touched by a patch, filtered by extension.

    Strategy 1: git apply --numstat  (preferred)
    Strategy 2: parse diff --git headers  (fallback)
    """
    clean = patch_text.replace("\r\n", "\n").replace("\r", "\n")
    fd, tmp = tempfile.mkstemp(suffix=".diff")
    os.close(fd)
    tmp_path = Path(tmp)
    tmp_path.write_text(clean, encoding="utf-8")

    files = []

    # Strategy 1 — git apply --numstat
    r = subprocess.run(
        f'git apply --numstat "{tmp_path}"',
        cwd=repo_path, shell=True, text=True, capture_output=True
    )
    if r.returncode == 0:
        for line in r.stdout.splitlines():
            parts = line.split("\t")
            if len(parts) == 3:
                p = parts[2].replace("\\", "/").lstrip("ab/")
                if Path(p).suffix in exts:
                    files.append(p)

    # Strategy 2 — parse diff headers (fallback)
    if not files:
        print("  [changed_files] numstat failed, falling back to diff header parse")
        for line in clean.splitlines():
            if line.startswith("diff --git "):
                m = re.match(r"diff --git a/(.+) b/(.+)", line)
                if m:
                    for p in (m.group(1), m.group(2)):
                        if Path(p).suffix in exts:
                            files.append(p)

    tmp_path.unlink(missing_ok=True)
    files = sorted(set(files))
    print(f"  [changed_files] {len(files)} file(s): {files[:5]}")
    return files


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst = valid_instances[0]
exts = LANG_EXT[inst["lang"]]

print("Step 4 — Detecting changed files")
print("  [gold patch]")
gold_files = get_changed_files(repo_path, inst["gold_patch_text"], exts)

print("  [llm patch]")
llm_files = []
if inst["llm_patch_path"].exists():
    llm_patch_text = inst["llm_patch_path"].read_text(encoding="utf-8", errors="ignore")
    llm_files = get_changed_files(repo_path, llm_patch_text, exts)
else:
    print("  no _patch.diff found")

Step 4 — Detecting changed files
  [gold patch]
  [changed_files] 1 file(s): ['qutebrowser/qt/machinery.py']
  [llm patch]
  [changed_files] 1 file(s): ['qutebrowser/qt/machinery.py']


## Step 4b — Filter Patch


In [38]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 4b — Filter Patch (remove test / docs / repro hunks)
# ═══════════════════════════════════════════════════════════════════════════

# ── Flag ───────────────────────────────────────────────────────────────────
FILTER_PATCHES = True   # set to False to apply patches as-is

import re as _re


class PatchCleaner:
    """
    Cleans git diffs by removing newly-added top-level files and
    files matching known non-production patterns (tests, docs, CI, etc.).
    """

    AUXILIARY_PATTERNS = [
    r'^IMPLEMENTATION_SUMMARY\.md$',
    r'^demo_.*\.py$',
    r'^example_.*\.py$',
    r'^implement_.*\.py$',
    r'^setup_.*\.py$',
    r'^[^/]+_demo\.py$',
    r'^[^/]+_example\.py$',
    r'^[^/]+_implementation\.py$',
    r'^test_implementation\.py$',
    r'^verify_.*\.py$',
    r'^validation_.*\.py$',
    r'^check_.*\.py$',
    r'^run_.*\.py$',
    r'^quick_test\.py$',
    r'^usage_.*\.py$',
    r'^[^/]+\.txt$',
    r'^notes\.md$',
    r'^TODO\.md$',
    r'^NOTES\.md$',
    # Test files — match at any depth
    r"(^|/)test_.*\.py$",
    r"(^|/).*_test\.py$",
    r"(^|/)tests?/",              # tests/ or test/ directories
    r"(^|/)tests\.py$",
    r"(^|/)test\.py$",
    r"(^|/)test_implementation\.py$",
    r"(^|/)final_validation\.py$",
    r"(^|/)validation\.py$",
    # doc/docs directories
    r"(^|/)docs?/",
    # documentation file extensions
    r"\.(md|rst|txt|adoc|asciidoc)$",
    # YAML files
    r"\.(yml|yaml)$",
]

    def __init__(self):
        self._aux_regex = [_re.compile(p, _re.IGNORECASE) for p in self.AUXILIARY_PATTERNS]

    def is_auxiliary_file(self, filepath: str) -> bool:
        """Return True if the path matches any non-production pattern."""
        p = filepath.replace("\\", "/").lstrip("ab/")
        return any(pat.search(p) for pat in self._aux_regex)

    def is_top_level_file(self, filepath: str) -> bool:
        """Return True if the file lives directly in the repo root (no slashes)."""
        return "/" not in filepath.replace("\\", "/").lstrip("ab/")

    def _extract_path(self, header: str) -> str | None:
        """Extract the b/-side path from a `diff --git` header."""
        m = _re.match(r"^diff --git a/(.+) b/(.+)$", header)
        return m.group(2) if m else None

    def clean_diff(self, patch_text: str) -> tuple[str, list[str], list[str]]:
        """
        Split a unified diff into kept and dropped file blocks.

        A file block is dropped when EITHER condition holds:
          • it is a *newly added* file at the repo root, OR
          • its path matches any entry in AUXILIARY_PATTERNS.

        Returns
        -------
        filtered_patch : str
        kept_files     : list[str]
        dropped_files  : list[str]
        """
        lines = patch_text.replace("\r\n", "\n").replace("\r", "\n").splitlines(keepends=True)

        # Split into per-file blocks on `diff --git` boundaries
        blocks: list[list[str]] = []
        current: list[str] = []
        for line in lines:
            if line.startswith("diff --git ") and current:
                blocks.append(current)
                current = []
            current.append(line)
        if current:
            blocks.append(current)

        kept_files: list[str] = []
        dropped_files: list[str] = []
        kept_blocks: list[list[str]] = []

        for block in blocks:
            header = block[0].rstrip("\n")
            file_path = self._extract_path(header)

            if file_path is None:
                # Malformed header — keep to be safe
                kept_blocks.append(block)
                continue

            is_new_file = any("new file mode" in line for line in block[:6])

            if (is_new_file and self.is_top_level_file(file_path)) or \
               self.is_auxiliary_file(file_path):
                dropped_files.append(file_path)
            else:
                kept_files.append(file_path)
                kept_blocks.append(block)

        filtered_patch = "".join("".join(b) for b in kept_blocks)
        return filtered_patch, kept_files, dropped_files


# ── Module-level helpers (same interface as before) ─────────────────────────

_cleaner = PatchCleaner()


def filter_patch(patch_text: str) -> tuple[str, list[str], list[str]]:
    """Thin wrapper around PatchCleaner.clean_diff() — keeps the original signature."""
    return _cleaner.clean_diff(patch_text)


def maybe_filter_patch(patch_text: str, label: str = "") -> str:
    """
    Apply filter_patch() if FILTER_PATCHES is True, otherwise return as-is.
    Prints a summary of what was kept and dropped.
    """
    if not FILTER_PATCHES:
        return patch_text

    filtered, kept, dropped = filter_patch(patch_text)

    tag = f"[{label}] " if label else ""
    print(f"  {tag}patch filter: kept {len(kept)} file(s), dropped {len(dropped)} file(s)")
    if dropped:
        for f in dropped:
            print(f"    ✗ dropped: {f}")
    if kept:
        for f in kept:
            print(f"    ✓ kept:    {f}")

    if not kept:
        print(f"  {tag}WARNING: all hunks were filtered out — patch is empty")

    return filtered


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst = valid_instances[0]

print("Step 4b — Filtering patches")
print("\n  [gold patch]")
gold_patch_filtered = maybe_filter_patch(inst["gold_patch_text"], label="gold")

print("\n  [llm patch]")
llm_patch_filtered = ""
if inst["llm_patch_path"].exists():
    llm_raw = inst["llm_patch_path"].read_text(encoding="utf-8", errors="ignore")
    llm_patch_filtered = maybe_filter_patch(llm_raw, label="llm")

Step 4b — Filtering patches

  [gold patch]
  [gold] patch filter: kept 1 file(s), dropped 0 file(s)
    ✓ kept:    qutebrowser/qt/machinery.py

  [llm patch]
  [llm] patch filter: kept 1 file(s), dropped 0 file(s)
    ✓ kept:    qutebrowser/qt/machinery.py


In [39]:
def maybe_filter_patch(patch_text: str, save_path: Path = None, label: str = "") -> str:
    """
    Apply filter_patch() if FILTER_PATCHES is True, otherwise return as-is.
    If save_path is given, write the filtered patch to that file.
    Prints a summary of what was kept and dropped.
    """
    if not FILTER_PATCHES:
        return patch_text

    filtered, kept, dropped = filter_patch(patch_text)

    tag = f"[{label}] " if label else ""
    print(f"  {tag}patch filter: kept {len(kept)} file(s), dropped {len(dropped)} file(s)")
    if dropped:
        for f in dropped:
            print(f"    ✗ dropped: {f}")
    if kept:
        for f in kept:
            print(f"    ✓ kept:    {f}")

    if not kept:
        print(f"  {tag}WARNING: all hunks were filtered out — patch is empty")

    if save_path is not None:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        save_path.write_text(filtered, encoding="utf-8")
        print(f"  {tag}filtered patch saved → {save_path}")

    return filtered


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst         = valid_instances[0]
inst_dir     = inst["inst_dir"]          # same folder as _patch.diff

print("Step 4b — Filtering patches")
print("\n  [gold patch]")
gold_patch_filtered = maybe_filter_patch(
    inst["gold_patch_text"],
    save_path=inst_dir / "filtered_gold.diff",
    label="gold",
)

print("\n  [llm patch]")
llm_patch_filtered = ""
if inst["llm_patch_path"].exists():
    llm_raw = inst["llm_patch_path"].read_text(encoding="utf-8", errors="ignore")
    llm_patch_filtered = maybe_filter_patch(
        llm_raw,
        save_path=inst_dir / "filtered_llm.diff",
        label="llm",
    )


Step 4b — Filtering patches

  [gold patch]
  [gold] patch filter: kept 1 file(s), dropped 0 file(s)
    ✓ kept:    qutebrowser/qt/machinery.py
  [gold] filtered patch saved → /home/AV00500/Issam/SWE-Bench_Pro/swebench_pro_pipeline/data/claude-45sonnet-10132025/eval/instance_qutebrowser__qutebrowser-a25e8a09873838ca9efefd36ea8a45170bbeb95c-vc2f56a753b62a190ddb23cd330c257b9cf560d12/filtered_gold.diff

  [llm patch]
  [llm] patch filter: kept 1 file(s), dropped 0 file(s)
    ✓ kept:    qutebrowser/qt/machinery.py
  [llm] filtered patch saved → /home/AV00500/Issam/SWE-Bench_Pro/swebench_pro_pipeline/data/claude-45sonnet-10132025/eval/instance_qutebrowser__qutebrowser-a25e8a09873838ca9efefd36ea8a45170bbeb95c-vc2f56a753b62a190ddb23cd330c257b9cf560d12/filtered_llm.diff


## Step 5 — Export Metrics from a UDB
Run `und metrics`, then filter the output CSV to keep only File, Class, and Function entities that belong to the changed files.

In [40]:
import csv

METRICS = [
    "AvgCountLine", "AvgCountLineBlank", "AvgCountLineCode", "AvgCountLineComment",
    "AvgCyclomatic", "AvgCyclomaticModified", "AvgCyclomaticStrict", "AvgCyclomaticStrictModified",
    "AvgEssential", "CCViolDensityCode", "CCViolDensityLine", "CountCCViol",
    "CountCCViolType", "CountClassBase", "CountClassCoupled", "CountClassCoupledModified",
    "CountClassDerived", "CountDeclClass", "CountDeclClassMethod", "CountDeclClassVariable",
    "CountDeclExecutableUnit", "CountDeclFile", "CountDeclFunction", "CountDeclInstanceMethod",
    "CountDeclInstanceVariable", "CountDeclMethod", "CountDeclMethodAll", "CountDeclMethodDefault",
    "CountDeclMethodPrivate", "CountDeclMethodProtected", "CountDeclMethodPublic", "CountInput",
    "CountLine", "CountLineBlank", "CountLineCode", "CountLineCodeDecl",
    "CountLineCodeExe", "CountLineComment", "CountOutput", "CountPath",
    "CountPathLog", "CountSemicolon", "CountStmt", "CountStmtDecl",
    "CountStmtExe", "Cyclomatic", "CyclomaticModified", "CyclomaticStrict",
    "CyclomaticStrictModified", "Essential", "MaxCyclomatic", "MaxCyclomaticModified",
    "MaxCyclomaticStrict", "MaxCyclomaticStrictModified", "MaxEssential", "MaxInheritanceTree",
    "MaxNesting", "PercentLackOfCohesion", "PercentLackOfCohesionModified", "RatioCommentToCode",
    "SumCyclomatic", "SumCyclomaticModified", "SumCyclomaticStrict", "SumCyclomaticStrictModified",
    "SumEssential",
]

# Maps Understand Kind substrings → normalized Level (order matters)
LEVEL_KEYWORDS = [
    ("File",     "File"),
    ("Class",    "Class"),
    ("Function", "Function"),
    ("Method",   "Function"),
]


def export_metrics(udb_path, changed_files, out_csv):
    """
    Run `und metrics` on udb_path, then filter the resulting CSV to keep
    only File, Class, and Function entities that belong to changed_files.

    For File entities:            Name  column = relative path
    For Class/Function entities:  File  column = declaring file path (may be absolute)
    """
    print(f"  [export_metrics] running und metrics on {udb_path.name}...")
    run_und(["metrics", str(udb_path)], timeout=300)

    metrics_csv = udb_path.with_suffix(".csv")
    if not metrics_csv.exists():
        print(f"  [export_metrics] WARNING: no CSV produced at {metrics_csv}")
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(columns=["Entity", "Kind", "Level", "File"] + METRICS).to_csv(out_csv, index=False)
        return

    with open(metrics_csv, encoding="utf-8", errors="replace") as f:
        all_rows = list(csv.DictReader(f))
    print(f"  [export_metrics] {len(all_rows)} total rows in UDB metrics CSV")

    tail_set = {p.replace("\\", "/") for p in changed_files}

    rows = []
    for r in all_rows:
        kind       = r.get("Kind", "")
        name       = r.get("Name", "")
        decl_file  = r.get("File", "").replace("\\", "/")
        uniquename = r.get("Entity_Uniquename", "").replace("\\", "/")

        level = None
        for keyword, lvl in LEVEL_KEYWORDS:
            if keyword in kind:
                level = lvl
                break
        if level is None:
            continue

        file_path = name.replace("\\", "/") if level == "File" else decl_file

        matched = any(t in uniquename for t in tail_set)
        if not matched and file_path:
            matched = any(
                file_path == t or
                file_path.endswith("/" + t) or
                file_path.endswith(t)
                for t in tail_set
            )
        if not matched:
            continue

        tail_set = {p.replace("\\", "/") for p in changed_files}

        # DEBUG
        print(f"  [export_metrics] tail_set: {tail_set}")
        if all_rows:
            sample = all_rows[0]
            print(f"  [export_metrics] sample row — Name={sample.get('Name','')!r}  File={sample.get('File','')!r}  Uniquename={sample.get('Entity_Uniquename','')!r}")

        row = {"Entity": name, "Kind": kind, "Level": level, "File": file_path}
        for m in METRICS:
            row[m] = r.get(m, "")
        rows.append(row)

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows, columns=["Entity", "Kind", "Level", "File"] + METRICS).to_csv(out_csv, index=False)
    print(f"  [export_metrics] {len(rows)} matched rows → {out_csv.name}")
    print(f"    File={sum(1 for r in rows if r['Level']=='File')}  "
          f"Class={sum(1 for r in rows if r['Level']=='Class')}  "
          f"Function={sum(1 for r in rows if r['Level']=='Function')}")


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst    = valid_instances[0]
out_dir = OUT_DIR / inst["repo_name"] / inst["iid"]
out_dir.mkdir(parents=True, exist_ok=True)

print("Step 5 — Exporting BEFORE metrics (from base UDB copies)")
if gold_files:
    export_metrics(gold_udb, gold_files, out_dir / "before_gold.csv")
if llm_files:
    export_metrics(llm_udb,  llm_files,  out_dir / "before_llm.csv")

Step 5 — Exporting BEFORE metrics (from base UDB copies)
  [export_metrics] running und metrics on qutebrowser_gold.und...
  [und metrics] understand_dbs/qutebrowser_gold.und
  → OK
  [export_metrics] 8363 total rows in UDB metrics CSV
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}

## Step 6 — Apply Patch and Export AFTER Metrics

Each patch is applied to the **shared repo** and analyzed against its **own UDB copy**.
No revert is needed — after both patches are processed the repo is reset to base commit
and the UDB copies are discarded.

In [41]:
def apply_patch(repo_path, patch_text):
    """
    Apply a patch using three strategies in order.
    Returns the temp diff Path on success, or None on failure.
    """
    clean = patch_text.replace("\r\n", "\n").replace("\r", "\n")
    fd, tmp = tempfile.mkstemp(suffix=".diff")
    os.close(fd)
    tmp_path = Path(tmp)
    tmp_path.write_text(clean, encoding="utf-8")

    for cmd in (
        f'git apply "{tmp_path}"',
        f'git apply --ignore-space-change --ignore-whitespace "{tmp_path}"',
        f'git apply --3way "{tmp_path}"',
    ):
        r = subprocess.run(cmd, cwd=repo_path, shell=True, text=True, capture_output=True)
        if r.returncode == 0:
            print(f"  [apply] OK")
            return tmp_path

    print(f"  [apply] FAILED — all strategies exhausted")
    tmp_path.unlink(missing_ok=True)
    return None


def reset_repo(repo_path):
    """Hard-reset the repo to discard any applied patch."""
    subprocess.run(["git", "reset", "--hard"], cwd=repo_path,
                   text=True, capture_output=True)
    subprocess.run(["git", "clean", "-fd"],    cwd=repo_path,
                   text=True, capture_output=True)
    print(f"  [git] repo reset to HEAD")


def process_patch(repo_path, udb_path, patch_text, changed_files, out_dir, kind):
    """
    Apply patch → und analyze -changed → export AFTER metrics.

    No revert is needed because each patch has its own UDB copy.
    The repo is reset to HEAD after each patch so the next patch
    starts from a clean working tree.

    Parameters
    ----------
    repo_path     : Path  — shared local repo clone
    udb_path      : Path  — the patch-specific UDB copy (gold or llm)
    patch_text    : str   — raw unified diff
    changed_files : list  — relative paths of files touched by the patch
    out_dir       : Path  — output directory for this instance
    kind          : str   — "gold" or "llm"
    """
    if not changed_files:
        print(f"  [{kind}] no changed files — skipping")
        return

    print(f"\n── [{kind.upper()}] APPLY ──")
    tmp_path = apply_patch(repo_path, patch_text)
    if tmp_path is None:
        print(f"  [{kind}] patch failed — skipping AFTER metrics")
        return

    try:
        print(f"\n── [{kind.upper()}] AFTER ──")
        # und detects which files changed on disk since last analysis
        print(f"  [analyze] incremental analysis (-changed) on {udb_path.name}...")
        run_und(["analyze", "-changed", str(udb_path)], timeout=600)
        export_metrics(udb_path, changed_files, out_dir / f"after_{kind}.csv")
    finally:
        # Reset repo so the next patch starts from a clean working tree
        reset_repo(repo_path)
        tmp_path.unlink(missing_ok=True)


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst    = valid_instances[0]
out_dir = OUT_DIR / inst["repo_name"] / inst["iid"]
out_dir.mkdir(parents=True, exist_ok=True)

print("Step 6 — Apply patches and export AFTER metrics")

# Gold patch — uses gold_udb copy
process_patch(repo_path, gold_udb, gold_patch_filtered, gold_files, out_dir, "gold")

# LLM patch — uses llm_udb copy (repo is already reset after gold)
if inst["llm_patch_path"].exists():
    llm_patch_text = inst["llm_patch_path"].read_text(encoding="utf-8", errors="ignore")
    process_patch(repo_path, llm_udb, llm_patch_filtered, llm_files, out_dir, "llm")


Step 6 — Apply patches and export AFTER metrics

── [GOLD] APPLY ──
  [apply] OK

── [GOLD] AFTER ──
  [analyze] incremental analysis (-changed) on qutebrowser_gold.und...
  [und analyze] -changed understand_dbs/qutebrowser_gold.und
  → OK
  [export_metrics] running und metrics on qutebrowser_gold.und...
  [und metrics] understand_dbs/qutebrowser_gold.und
  → OK
  [export_metrics] 8361 total rows in UDB metrics CSV
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machinery.py'}
  [export_metrics] sample row — Name='AutoFileTests'  File=''  Uniquename='@lAutoFileTests@kyAutoFileTests:global'
  [export_metrics] tail_set: {'qutebrowser/qt/machine

## Step 7 — Compute Before/After Diff
For each patch, compute the metric delta (after − before) per entity and write a diff CSV.

In [17]:
def compute_diff(before_csv, after_csv, diff_csv):
    """
    Compute metric deltas between before and after CSVs.
    Each output row contains before_<M>, after_<M>, diff_<M> for every metric.
    """
    print(f"  [diff] {before_csv.name} vs {after_csv.name}")

    def load(p):
        return pd.read_csv(p).to_dict(orient="records") if p.exists() and p.stat().st_size else []

    b_rows, a_rows = load(before_csv), load(after_csv)
    b_dict = {(r.get("Entity", ""), r.get("Kind", "")): r for r in b_rows}
    a_dict = {(r.get("Entity", ""), r.get("Kind", "")): r for r in a_rows}

    diff_rows = []
    for entity, kind in sorted(set(b_dict) | set(a_dict)):
        b, a = b_dict.get((entity, kind), {}), a_dict.get((entity, kind), {})
        level  = b.get("Level") or a.get("Level")
        status = ("new"     if (entity, kind) not in b_dict else
                  "removed" if (entity, kind) not in a_dict else "modified")

        row = {"Entity": entity, "Kind": kind, "Level": level,
               "status": status, "File": b.get("File") or a.get("File")}
        for m in METRICS:
            vb = float(b.get(m) or 0)
            va = float(a.get(m) or 0)
            row[f"before_{m}"] = vb
            row[f"after_{m}"]  = va
            row[f"diff_{m}"]   = va - vb
        diff_rows.append(row)

    cols = ["Entity", "Kind", "Level", "status", "File"] + [
        f"{prefix}_{m}" for m in METRICS for prefix in ("before", "after", "diff")
    ]
    diff_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(diff_rows, columns=cols).to_csv(diff_csv, index=False)
    print(f"  [diff] {len(diff_rows)} rows → {diff_csv.name}")
    lvl = pd.DataFrame(diff_rows)
    if not lvl.empty:
        print(f"    File={lvl[lvl.Level=='File'].shape[0]}  "
              f"Class={lvl[lvl.Level=='Class'].shape[0]}  "
              f"Function={lvl[lvl.Level=='Function'].shape[0]}")


# ── Smoke-test ─────────────────────────────────────────────────────────────
inst    = valid_instances[0]
out_dir = OUT_DIR / inst["repo_name"] / inst["iid"]

print("Step 7 — Computing diffs")
for kind in ("gold", "llm"):
    b = out_dir / f"before_{kind}.csv"
    a = out_dir / f"after_{kind}.csv"
    d = out_dir / f"diff_{kind}.csv"
    if b.exists() and a.exists():
        compute_diff(b, a, d)
        b.unlink(missing_ok=True)   # clean up intermediate files
        a.unlink(missing_ok=True)

Step 7 — Computing diffs
  [diff] before_gold.csv vs after_gold.csv
  [diff] 185 rows → diff_gold.csv
    File=3  Class=21  Function=161
  [diff] before_llm.csv vs after_llm.csv
  [diff] 121 rows → diff_llm.csv
    File=3  Class=9  Function=109


## Step 8 — Full Loop Over All Instances
Run the complete pipeline over every valid instance.

In [47]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 8a — Pre-build base UDBs (sequential, one per repo)
# ═══════════════════════════════════════════════════════════════════════════
print("Step 8a — Building base UDBs (sequential)...")

BASE_UDBS = {}   # repo_name → Path to base .und

for repo_id, repo_name in REPO_MAP.items():
    lang          = REPO_LANG[repo_id]
    base_udb      = UDB_DIR / f"{repo_name}_base.und"
    base_repo     = clone_repo_if_needed(repo_id, repo_name)
    repo_instances = [i for i in valid_instances if i["repo_name"] == repo_name]
    if not repo_instances:
        continue

    if base_udb.exists():
        print(f"  [{repo_name}] base UDB already exists — skipping")
    else:
        first_commit = repo_instances[0]["base_commit"]
        print(f"  [{repo_name}] building base UDB at commit {first_commit[:10]}...")
        checkout_commit(base_repo, first_commit)
        create_base_udb(base_udb, base_repo, lang)

    BASE_UDBS[repo_name] = base_udb

print(f"\nBase UDBs ready: {list(BASE_UDBS.keys())}")


Step 8a — Building base UDBs (sequential)...
  [clone] already exists: /home/AV00500/Issam/SWE-Bench_Pro/repos/ansible
  [clone] already exists: /home/AV00500/Issam/SWE-Bench_Pro/repos/qutebrowser
  [qutebrowser] base UDB already exists — skipping
  [clone] already exists: /home/AV00500/Issam/SWE-Bench_Pro/repos/openlibrary

Base UDBs ready: ['qutebrowser']


In [45]:
# ═══════════════════════════════════════════════════════════════════════════
# Step 8 — Pipeline: 3 parallel repo-workers, sequential within each repo
# ═══════════════════════════════════════════════════════════════════════════
import os
import sys
import shutil
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# ── Config ─────────────────────────────────────────────────────────────────
TEST_MODE  = False  # True = process only first instance per repo
N_WORKERS  = 3      # one worker per repo — matches number of repos
LOG_DIR    = OUT_DIR / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
MASTER_LOG = LOG_DIR / "pipeline.log"


class _Tee:
    """
    Redirect all print() calls inside a worker to a log file.
    stdout is also preserved so tqdm progress is visible in the notebook.
    """
    def __init__(self, path: Path):
        self._path = path
        self._file = None
        self._orig_stdout = None
        self._orig_stderr = None

    def write(self, msg):
        self._file.write(msg)
        self._file.flush()
        self._orig_stdout.write(msg)

    def flush(self):
        self._file.flush()
        self._orig_stdout.flush()

    def __enter__(self):
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._file = open(self._path, "w", encoding="utf-8", buffering=1)
        self._orig_stdout = sys.stdout
        self._orig_stderr = sys.stderr
        sys.stdout = self
        sys.stderr = self
        return self

    def __exit__(self, *_):
        sys.stdout = self._orig_stdout
        sys.stderr = self._orig_stderr
        self._file.close()


def _make_worker_repo(base_repo: Path, repo_name: str) -> Path:
    """
    Create one isolated worker clone per repo using git clone --local.
    Called once per repo-worker (not per instance).
    Handles submodules correctly and uses hardlinks for speed.
    """
    worker_repo = REPO_ROOT / f"{repo_name}_worker_{os.getpid()}"
    if worker_repo.exists():
        return worker_repo

    print(f"  [repo] git clone --local {repo_name} → {worker_repo.name}")
    r = subprocess.run(
        ["git", "clone", "--local", str(base_repo), str(worker_repo)],
        text=True, capture_output=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"git clone --local failed:\n{r.stderr}")

    r2 = subprocess.run(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=worker_repo, text=True, capture_output=True,
    )
    if r2.returncode != 0:
        print(f"  [repo] WARNING: submodule init failed:\n{r2.stderr[:200]}")

    print(f"  [repo] OK → {worker_repo}")
    return worker_repo


def _process_single_instance(inst, repo_path, repo_log_dir):
    """
    Run the full pipeline for one instance.
    repo_path is the shared worker clone for this repo.
    Called sequentially inside _process_repo_worker.
    """
    iid      = inst["iid"]
    repo_name = inst["repo_name"]
    lang     = inst["lang"]
    exts     = LANG_EXT[lang]
    out_dir  = OUT_DIR / repo_name / iid
    out_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*60}")
    print(f"Instance: {iid}")

    # Skip if already done
    if (out_dir / "diff_gold.csv").exists() and (out_dir / "diff_llm.csv").exists():
        print(f"  already done — skipping")
        return {"iid": iid, "status": "skipped", "error": None}

    try:
        # ── Checkout this instance's commit ───────────────────────────────
        checkout_commit(repo_path, inst["base_commit"])

        # ── Per-instance UDB copies from pre-built base ───────────────────
        safe_iid = iid.replace("/", "_")
        gold_udb = UDB_DIR / f"{safe_iid}_gold.und"
        llm_udb  = UDB_DIR / f"{safe_iid}_llm.und"
        base_udb = BASE_UDBS[repo_name]
        print(f"\n── UDB COPY (from pre-built base) ──")
        copy_udb(base_udb, gold_udb)
        copy_udb(base_udb, llm_udb)

        # ── Detect changed files ──────────────────────────────────────────
        gold_patch_text = inst["gold_patch_text"]
        llm_text        = ""
        gold_files, llm_files = [], []

        if gold_patch_text.strip():
            gold_files = get_changed_files(repo_path, gold_patch_text, exts)

        if inst["llm_patch_path"].exists():
            llm_text  = inst["llm_patch_path"].read_text(encoding="utf-8", errors="ignore")
            llm_files = get_changed_files(repo_path, llm_text, exts)

        # ── Filter patches ────────────────────────────────────────────────
        patch_dir = inst["llm_patch_path"].parent
        if FILTER_PATCHES:
            gold_patch_text = maybe_filter_patch(
                gold_patch_text,
                save_path=patch_dir / "filtered_gold.diff",
                label="gold",
            )
            if llm_text:
                llm_text = maybe_filter_patch(
                    llm_text,
                    save_path=patch_dir / "filtered_llm.diff",
                    label="llm",
                )

        # ── BEFORE metrics ────────────────────────────────────────────────
        print(f"\n── BEFORE METRICS ──")
        if gold_files:
            export_metrics(gold_udb, gold_files, out_dir / "before_gold.csv")
        if llm_files:
            export_metrics(llm_udb, llm_files, out_dir / "before_llm.csv")

        # ── Apply patches + AFTER metrics ─────────────────────────────────
        print(f"\n── APPLY + AFTER ──")
        if not (out_dir / "diff_gold.csv").exists():
            process_patch(repo_path, gold_udb, gold_patch_text, gold_files, out_dir, "gold")
        if not (out_dir / "diff_llm.csv").exists() and llm_text:
            process_patch(repo_path, llm_udb, llm_text, llm_files, out_dir, "llm")

        # ── Compute diffs ─────────────────────────────────────────────────
        print(f"\n── DIFFS ──")
        for kind in ("gold", "llm"):
            b = out_dir / f"before_{kind}.csv"
            a = out_dir / f"after_{kind}.csv"
            d = out_dir / f"diff_{kind}.csv"
            if b.exists() and a.exists():
                compute_diff(b, a, d)
                b.unlink(missing_ok=True)
                a.unlink(missing_ok=True)

        # ── Cleanup per-instance UDBs (keep worker repo for next instance) ─
        for udb in (gold_udb, llm_udb):
            shutil.rmtree(str(udb), ignore_errors=True)
            udb.with_suffix(".csv").unlink(missing_ok=True)

        print(f"\n[DONE] {iid}")
        return {"iid": iid, "status": "done", "error": None}

    except Exception as e:
        tb = traceback.format_exc()
        print(f"\n[ERROR] {iid}\n{tb}")
        return {"iid": iid, "status": "error", "error": str(e)}


def _process_repo_worker(repo_name: str, repo_instances: list) -> list:
    """
    Worker function for one repo.
    Runs in a subprocess. Processes all instances of the repo SEQUENTIALLY.
    Returns a list of per-instance result dicts.
    """
    repo_log = LOG_DIR / f"{repo_name}_worker.log"
    results  = []

    with _Tee(repo_log):
        print(f"{'#'*60}")
        print(f"REPO WORKER: {repo_name}  ({len(repo_instances)} instances)  PID={os.getpid()}")
        print(f"{'#'*60}")

        try:
            # One shared worker clone for all instances of this repo
            repo_id   = repo_instances[0]["repo_id"]
            base_repo = clone_repo_if_needed(repo_id, repo_name)
            repo_path = _make_worker_repo(base_repo, repo_name)

        except Exception as e:
            tb = traceback.format_exc()
            print(f"[FATAL] Could not set up worker repo for {repo_name}:\n{tb}")
            return [{"iid": inst["iid"], "status": "error", "error": str(e)}
                    for inst in repo_instances]

        # Process instances one by one
        for inst in repo_instances:
            res = _process_single_instance(inst, repo_path, LOG_DIR / repo_name)
            results.append(res)

        # Cleanup shared worker repo after all instances are done
        shutil.rmtree(str(repo_path), ignore_errors=True)
        print(f"\n[REPO DONE] {repo_name} — cleaned up worker repo")

    return results


# ── Group instances by repo ────────────────────────────────────────────────
from collections import defaultdict
repo_groups = defaultdict(list)
instances_to_run = valid_instances[:1] if TEST_MODE else valid_instances
for inst in instances_to_run:
    repo_groups[inst["repo_name"]].append(inst)

print(f"Step 8 — Pipeline: {len(instances_to_run)} instances across {len(repo_groups)} repos")
print(f"  Workers: {N_WORKERS} (one per repo, sequential within each repo)")
for rname, rinsts in repo_groups.items():
    print(f"  {rname}: {len(rinsts)} instances")
print(f"  Logs: {LOG_DIR}")
print(f"  Master log: {MASTER_LOG}")

# ── Run 3 repo-workers in parallel ────────────────────────────────────────
all_results = []
with open(MASTER_LOG, "w", encoding="utf-8", buffering=1) as master_log:
    master_log.write(f"Pipeline started — {len(instances_to_run)} instances, "
                     f"{len(repo_groups)} repo-workers\n\n")

    with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
        futures = {
            pool.submit(_process_repo_worker, repo_name, repo_insts): repo_name
            for repo_name, repo_insts in repo_groups.items()
        }

        for future in tqdm(as_completed(futures), total=len(futures), desc="repos"):
            repo_name = futures[future]
            try:
                repo_results = future.result()
            except Exception as e:
                tb = traceback.format_exc()
                repo_results = [{"iid": f"{repo_name}/?", "status": "error", "error": str(e)}]

            for res in repo_results:
                all_results.append(res)
                status_line = f"[{res['status'].upper():8s}] {res['iid']}"
                if res["error"]:
                    status_line += f"  — {res['error']}"
                print(status_line)
                master_log.write(status_line + "\n")

# ── Summary ────────────────────────────────────────────────────────────────
done    = sum(1 for r in all_results if r["status"] == "done")
skipped = sum(1 for r in all_results if r["status"] == "skipped")
errors  = sum(1 for r in all_results if r["status"] == "error")
print(f"\nDone={done}  Skipped={skipped}  Errors={errors}")
print(f"Repo logs: {LOG_DIR}/<repo>_worker.log")
print(f"Master log: {MASTER_LOG}")


Step 8 — Pipeline: 1 instances across 1 repos
  Workers: 3 (one per repo, sequential within each repo)
  qutebrowser: 1 instances
  Logs: understand_results/claude-45sonnet-10132025/logs
  Master log: understand_results/claude-45sonnet-10132025/logs/pipeline.log


repos:   0%|          | 0/1 [00:00<?, ?it/s]

############################################################
REPO WORKER: qutebrowser  (1 instances)  PID=1446186
############################################################
  [clone] already exists: /home/AV00500/Issam/SWE-Bench_Pro/repos/qutebrowser
  [repo] git clone --local qutebrowser → qutebrowser_worker_1446186


  [repo] OK → /home/AV00500/Issam/SWE-Bench_Pro/repos/qutebrowser_worker_1446186

Instance: instance_qutebrowser__qutebrowser-a25e8a09873838ca9efefd36ea8a45170bbeb95c-vc2f56a753b62a190ddb23cd330c257b9cf560d12
  already done — skipping

[REPO DONE] qutebrowser — cleaned up worker repo


repos: 100%|██████████| 1/1 [00:00<00:00,  3.20it/s]

[SKIPPED ] instance_qutebrowser__qutebrowser-a25e8a09873838ca9efefd36ea8a45170bbeb95c-vc2f56a753b62a190ddb23cd330c257b9cf560d12

Done=0  Skipped=1  Errors=0
Repo logs: understand_results/claude-45sonnet-10132025/logs/<repo>_worker.log
Master log: understand_results/claude-45sonnet-10132025/logs/pipeline.log


## Step 9 — Aggregate Metrics
Summarise the diff CSVs into one flat CSV with per-level columns for gold and LLM patches.

In [ ]:
CLASS_METRICS = {
    "CountClassBase", "CountClassCoupled", "CountClassCoupledModified",
    "CountClassDerived", "CountDeclInstanceMethod", "CountDeclInstanceVariable",
    "CountDeclMethod", "CountDeclMethodAll", "CountDeclMethodDefault",
    "CountDeclMethodPrivate", "CountDeclMethodProtected", "CountDeclMethodPublic",
    "MaxInheritanceTree", "PercentLackOfCohesion", "PercentLackOfCohesionModified",
}
FUNCTION_METRICS = {
    "Cyclomatic", "CyclomaticModified", "CyclomaticStrict", "CyclomaticStrictModified",
    "Essential", "MaxNesting", "CountInput", "CountOutput", "CountPath", "CountPathLog",
}


# ── Aggregation semantics per metric ──────────────────────────────────────
SUM_METRICS = {
    "CountCCViol", "CountCCViolType",
    "CountClassBase", "CountClassCoupled", "CountClassCoupledModified", "CountClassDerived",
    "CountDeclClass", "CountDeclClassMethod", "CountDeclClassVariable",
    "CountDeclExecutableUnit", "CountDeclFile", "CountDeclFunction",
    "CountDeclInstanceMethod", "CountDeclInstanceVariable",
    "CountDeclMethod", "CountDeclMethodAll", "CountDeclMethodDefault",
    "CountDeclMethodPrivate", "CountDeclMethodProtected", "CountDeclMethodPublic",
    "CountInput", "CountLine", "CountLineBlank", "CountLineCode",
    "CountLineCodeDecl", "CountLineCodeExe", "CountLineComment",
    "CountOutput", "CountPath", "CountPathLog",
    "CountSemicolon", "CountStmt", "CountStmtDecl", "CountStmtExe",
    "SumCyclomatic", "SumCyclomaticModified",
    "SumCyclomaticStrict", "SumCyclomaticStrictModified", "SumEssential",
}

MAX_METRICS = {
    "MaxCyclomatic", "MaxCyclomaticModified",
    "MaxCyclomaticStrict", "MaxCyclomaticStrictModified",
    "MaxEssential", "MaxInheritanceTree", "MaxNesting",
}

MEAN_METRICS = {
    "AvgCountLine", "AvgCountLineBlank", "AvgCountLineCode", "AvgCountLineComment",
    "AvgCyclomatic", "AvgCyclomaticModified", "AvgCyclomaticStrict", "AvgCyclomaticStrictModified",
    "AvgEssential",
    "CCViolDensityCode", "CCViolDensityLine",
    "Cyclomatic", "CyclomaticModified", "CyclomaticStrict", "CyclomaticStrictModified",
    "Essential",
    "PercentLackOfCohesion", "PercentLackOfCohesionModified",
    "RatioCommentToCode",
}


def _agg_series(series: "pd.Series", metric: str) -> float:
    """Apply the correct aggregation for a metric's diff column."""
    s = series.dropna()
    if s.empty:
        return 0.0
    if metric in MAX_METRICS:
        return float(s.max())
    if metric in MEAN_METRICS:
        return float(s.mean())
    # Default: SUM_METRICS and anything not explicitly classified
    return float(s.sum())


def aggregate_instance_metrics(diff_csv_path, is_gold=False):
    """
    Aggregate diff metrics by File, Class, and Function levels.

    Aggregation semantics:
      - SUM   for additive count metrics
      - MEAN  for ratios, averages, and per-entity complexity scores
      - MAX   for worst-case metrics (MaxCyclomatic, MaxNesting, etc.)

    Column naming:
        diff_<Metric>_file_human     diff_<Metric>_file_llm
        diff_<Metric>_class_human    diff_<Metric>_class_llm
        diff_<Metric>_function_human diff_<Metric>_function_llm
    """
    suffix = "human" if is_gold else "llm"
    result = {f"diff_{m}_{lvl}_{suffix}": 0.0
              for m in METRICS for lvl in ("file", "class", "function")}

    if not diff_csv_path.exists() or diff_csv_path.stat().st_size == 0:
        return result

    df = pd.read_csv(diff_csv_path)
    masks = {
        "file":     df["Level"] == "File",
        "class":    df["Level"] == "Class",
        "function": df["Level"] == "Function",
    }

    for m in METRICS:
        col = f"diff_{m}"
        if col not in df.columns:
            continue
        lvl = "class" if m in CLASS_METRICS else "function" if m in FUNCTION_METRICS else "file"
        result[f"{col}_{lvl}_{suffix}"] = _agg_series(df.loc[masks[lvl], col], m)

    return result



# ── Aggregate all processed instances ─────────────────────────────────────
print("Step 9 — Aggregating metrics...")
agg_rows = []

for inst in valid_instances:
    out_dir = OUT_DIR / inst["repo_name"] / inst["iid"]
    if not out_dir.exists():
        continue
    row = {"instance_id": inst["iid"], "repo": inst["repo_name"]}
    row.update(aggregate_instance_metrics(out_dir / "diff_gold.csv", is_gold=True))
    row.update(aggregate_instance_metrics(out_dir / "diff_llm.csv",  is_gold=False))
    agg_rows.append(row)

if agg_rows:
    agg_path = OUT_DIR / "final_aggregated_metrics.csv"
    pd.DataFrame(agg_rows).to_csv(agg_path, index=False)
    print(f"  → {len(agg_rows)} rows aggregated → {agg_path}")
else:
    print("  → no rows to aggregate yet")